# 04 · Uncertainty Quantification

LightGBM with pinball loss gives per-prediction quantile estimates without a second model family. We fit at 0.1 / 0.9 for 80% prediction intervals and check empirical coverage on a held-out season.

In [ ]:
from gaffer.providers.historical_csv import HistoricalCsvProvider
from gaffer.providers.fpl_api import LiveFplApiProvider
from gaffer.services.prediction_service import build_training_set
from gaffer.models.quantile import LgbmQuantilePredictor

td = build_training_set(LiveFplApiProvider(), HistoricalCsvProvider())
X = td.X.dropna()
y = td.y.loc[X.index]
seasons = td.seasons.loc[X.index]

holdout = seasons.max()
train_mask = seasons != holdout
model = LgbmQuantilePredictor(n_estimators=300).fit(X[train_mask], y[train_mask])

In [ ]:
lower, upper = model.predict_interval(X[~train_mask], quantiles=(0.1, 0.9))
observed = y[~train_mask]
coverage = ((observed >= lower) & (observed <= upper)).mean()
print(f'Empirical 80% coverage: {coverage:.2%}')

In [ ]:
import matplotlib.pyplot as plt

sample = observed.sample(200, random_state=0).sort_values().reset_index(drop=True)
plt.fill_between(range(len(sample)), lower[sample.index], upper[sample.index], alpha=0.3)
plt.plot(sample.values, 'k.')
plt.title('80% intervals vs realised points (random sample)')
plt.show()

**Next:** `05_optimization_walkthrough.ipynb` — feed point + interval predictions into the MILP.